In [2]:
import os
import glob
import re
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
# from tqdm.auto import tqdm
import matplotlib.pyplot as plt


In [3]:
# -------------------------
# Data & preprocessing
# -------------------------
FEATURE_COLS_BASE = [
    "batt_voltage", "batt_current", "soc", "soh",
]  # + temperature_M1..M5 + cell_voltages_1..13

def read_csv_files(folder: str, pattern: str = "*.csv") -> List[pd.DataFrame]:
    files = sorted(glob.glob(os.path.join(folder, pattern)))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # ensure expected columns exist (you can adapt)
        dfs.append(df)
    return dfs

In [4]:
def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with the features we'll use per timestamp."""
    cols = FEATURE_COLS_BASE.copy()
    temp_cols = sorted([c for c in df.columns if c.startswith("temperature_M")])
    cell_cols = sorted([c for c in df.columns if c.startswith("cell_voltages_")])
    cols += temp_cols + cell_cols
    # Additional engineered features: cell voltage mean, max-min imbalance, voltage deltas
    out = df[cols].copy()
    # add imbalance features
    out["cell_mean"] = out[cell_cols].mean(axis=1)
    out["cell_maxmin"] = out[cell_cols].max(axis=1) - out[cell_cols].min(axis=1)
    # per-cell differences to pack mean
    for i, c in enumerate(cell_cols, start=1):
        out[f"cell_diff_{i}"] = out[c] - out["cell_mean"]
    return out

In [5]:
def sliding_windows_from_df(df: pd.DataFrame, window_s: int = 60, step_s: int = 10, time_col: str = "time"):
    """Create windows of given seconds (we require time steps to be in seconds and regular).
       Returns list of numpy arrays shape (window_len, n_features).
    """
    data = build_feature_matrix(df).reset_index(drop=True)
    t = df[time_col].values
    # compute indices by time: assume time is monotonic
    windows = []
    start_idx = 0
    n = len(df)
    while start_idx < n:
        # find end index where t[end] - t[start] >= window_s
        end_idx = start_idx
        while end_idx < n and (t[end_idx] - t[start_idx] < window_s):
            end_idx += 1
        if end_idx - start_idx >= 2:
            windows.append(data.iloc[start_idx:end_idx].to_numpy(dtype=float))
        # advance by step in seconds -> find next index with t >= t[start] + step_s
        next_time = t[start_idx] + step_s
        # locate next index
        next_idx = np.searchsorted(t, next_time, side="left")
        if next_idx <= start_idx:
            next_idx = start_idx + 1
        start_idx = next_idx
    return windows

In [6]:
# -------------------------
# Synthetic defect injectors
# -------------------------
def inject_high_resistance(window: np.ndarray, cell_index: int, severity: float = 0.1) -> np.ndarray:
    """
    Simulate high series resistance on a cell by increasing deltaV ~ I*R during current flow.
    window: (T, F) where one feature is batt_current and cell_voltages are present.
    severity: fractional extra voltage drop per amp, e.g., 0.1 -> 0.1V/A (tunable)
    """
    w = window.copy()
    # find batt_current column index: assume it's first or so; better to map by header externally.
    # For this generic injector we assume batt_current is column 1 and cell voltages are at tail.
    # But we'll not hardcode; instead user should pass column index mapping. For now assume:
    # batt_current = col 1, cell voltages start at col idx base_cell_idx
    # simplified: reduce targeted cell voltage by severity * current
    # Here we guess last 13 columns are cell voltages; adjust if necessary.
    T, F = w.shape
    # assume last 13 cols are cells
    n_cells = 13
    cell_cols_idx = list(range(F - n_cells, F))
    current_idx = 1  # batt_current position (based on build_feature_matrix earlier). Might be 1
    # apply extra drop when current>0 (charging) or current<0 (discharging) -> sign matters
    for i in range(T):
        I = w[i, current_idx]
        drop = severity * I
        # apply to chosen cell index (0-based)
        idx = cell_cols_idx[cell_index]
        w[i, idx] = w[i, idx] - drop
    return w


def inject_capacity_loss(window: np.ndarray, factor: float = 0.5) -> np.ndarray:
    """
    Simulate lower capacity by modifying SOC evolution: drop SOC by factor fraction across the window.
    """
    w = window.copy()
    # assume SOC is col 2 (based on build_feature_matrix ordering: batt_voltage, batt_current, soc,...)
    soc_idx = 2
    w[:, soc_idx] = w[:, soc_idx] * factor
    return w


def inject_persistent_imbalance(window: np.ndarray, cell_index: int, offset_v: float = 0.05) -> np.ndarray:
    """Add constant offset to a cell voltage across the entire window."""
    w = window.copy()
    T, F = w.shape
    n_cells = 13
    cell_cols_idx = list(range(F - n_cells, F))
    idx = cell_cols_idx[cell_index]
    w[:, idx] += offset_v
    return w